# 05 - Feature Engineering

**Goal**: Extract structured features from text that ML models can use directly.

**Learning concepts**: TF-IDF (term frequency–inverse document frequency), sentiment analysis,
regex-based keyword extraction, joining structured skill data.

**Research themes addressed**:
- Theme 1 (Salary): Do description length or sentiment predict salary?
- Theme 3 (Entry-Level Paradox): Do entry-level jobs demand senior-level skills?
- Theme 4 (Engagement): Do longer/more positive descriptions get more applications?

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer

from talentlens.config import (
    POSTINGS_NLP_PARQUET, POSTINGS_FEATURES_PARQUET, PROCESSED_DIR,
    JOB_SKILLS_CSV, SKILLS_MAP_CSV,
)
from talentlens.features import add_sentiment, add_senior_signals
from talentlens.plots import plot_top_categories, plot_box_by_category, save_fig

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)
%matplotlib inline

In [ ]:
# Load the NLP-preprocessed dataset from notebook 04
if not POSTINGS_NLP_PARQUET.exists():
    raise FileNotFoundError(
        "postings_nlp.parquet not found! "
        "Run notebook 04-mp-nlp-text-preprocessing.ipynb first."
    )

df = pd.read_parquet(POSTINGS_NLP_PARQUET)
print(f"Loaded {len(df):,} postings with NLP preprocessing")
df.info()

## 1. TF-IDF Analysis

**What is TF-IDF?** A way to measure how important a word is to a document relative to a whole collection.

- **TF** (Term Frequency): How often does this word appear in THIS document?
- **IDF** (Inverse Document Frequency): How rare is this word across ALL documents?
- **TF-IDF = TF × IDF**: Words that appear often in one document but rarely in others get high scores.

**Example**: "python" in a data science job description → high TF-IDF (specific to that job type).
"experience" in every job description → low TF-IDF (too common to be distinctive).

**Why we use the lemmatized text**: "engineer", "engineers", "engineering" would be 3 different terms.
After lemmatization, they're all "engineer" — giving a stronger, cleaner signal.

In [ ]:
# Fit TF-IDF on lemmatized descriptions
# max_features=5000: keep the top 5000 most important terms
# min_df=10: ignore terms that appear in fewer than 10 documents (too rare to generalize)
# max_df=0.95: ignore terms that appear in >95% of documents (too common to be distinctive)
tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=10,
    max_df=0.95,
    ngram_range=(1, 2),  # unigrams + bigrams ("machine learning", not just "machine")
)

tfidf_matrix = tfidf.fit_transform(df['desc_lemmatized'].fillna(''))
feature_names = tfidf.get_feature_names_out()

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]:,} documents × {tfidf_matrix.shape[1]:,} terms")
print(f"  → Sparse matrix density: {tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]) * 100:.2f}%")

### Why sparse?

Most documents only use a tiny fraction of the 5,000 terms, so the matrix is mostly zeros.
scikit-learn stores this as a **sparse matrix** — only the non-zero values are kept in memory.
If it were dense, 123K × 5000 × 8 bytes = ~4.6 GB. Sparse: just a few hundred MB.

In [ ]:
# Top TF-IDF terms across ALL documents
mean_tfidf = tfidf_matrix.mean(axis=0).A1  # .A1 converts sparse matrix → flat array
top_indices = mean_tfidf.argsort()[::-1][:30]

print("Top 30 TF-IDF terms (averaged across all job postings):")
print("-" * 50)
for i, idx in enumerate(top_indices, 1):
    print(f"  {i:2d}. {feature_names[idx]:<30s} {mean_tfidf[idx]:.4f}")

In [ ]:
# Top TF-IDF terms PER EXPERIENCE LEVEL
# This is where it gets interesting — what terms distinguish each level?
print("Top 10 TF-IDF terms per experience level:")
print("=" * 60)

exp_levels = ['Entry level', 'Mid-Senior level', 'Director', 'Executive']

for level in exp_levels:
    mask = df['experience_level'] == level
    if mask.sum() == 0:
        continue
    
    # Average TF-IDF for this experience level only
    level_mean = tfidf_matrix[mask.values].mean(axis=0).A1
    top_idx = level_mean.argsort()[::-1][:10]
    
    print(f"\n{level} ({mask.sum():,} postings):")
    for i, idx in enumerate(top_idx, 1):
        print(f"  {i:2d}. {feature_names[idx]:<30s} {level_mean[idx]:.4f}")

### Interpretation

- **Entry level**: Look for terms like "intern", "learning", "assist" — or do you see "experience", "manage"?
  If senior-sounding terms appear in entry-level TF-IDF, that supports the Entry-Level Paradox (Theme 3).
- **Director/Executive**: Expect "strategy", "leadership", "revenue", "growth" — business-oriented terms.
- **Mid-Senior level**: Should see the most technical terms — "python", "data", "engineering".

TF-IDF by group is a quick way to understand **what makes each category linguistically distinct**.

## 2. Sentiment Analysis

**What is sentiment analysis?** Measuring how positive, negative, or neutral text is.

TextBlob gives us two scores:
- **Polarity** (-1 to +1): Negative ↔ Positive. "Exciting opportunity" = positive; "demanding role" = less positive.
- **Subjectivity** (0 to 1): Factual ↔ Opinionated. "Must have Python" = objective; "Amazing team culture" = subjective.

**Hypothesis (Theme 4)**: More positive, enthusiastic descriptions may attract more applications.

**Limitation**: TextBlob is a simple rule-based model. It works for broad patterns but won't catch
subtle nuance. For production sentiment, you'd use a fine-tuned transformer model.

In [ ]:
# Add sentiment columns (this takes a few minutes on 123K texts)
df = add_sentiment(df, text_col='desc_clean')

print("\nSentiment statistics:")
print(df[['sentiment_polarity', 'sentiment_subjectivity']].describe().round(3))

In [ ]:
# Visualize sentiment distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['sentiment_polarity'], bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Sentiment Polarity Distribution')
axes[0].set_xlabel('Polarity (-1 = negative, +1 = positive)')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.5, label='Neutral')
axes[0].axvline(df['sentiment_polarity'].median(), color='orange', linestyle='--', 
                label=f"Median: {df['sentiment_polarity'].median():.3f}")
axes[0].legend()

sns.histplot(df['sentiment_subjectivity'], bins=50, ax=axes[1], color='coral')
axes[1].set_title('Sentiment Subjectivity Distribution')
axes[1].set_xlabel('Subjectivity (0 = factual, 1 = opinionated)')
axes[1].axvline(df['sentiment_subjectivity'].median(), color='orange', linestyle='--',
                label=f"Median: {df['sentiment_subjectivity'].median():.3f}")
axes[1].legend()

plt.tight_layout()
save_fig(fig, 'nlp_sentiment_distributions')
plt.show()

In [ ]:
# Sentiment by experience level
print("Median sentiment by experience level:")
sentiment_by_exp = df.groupby('experience_level')[['sentiment_polarity', 'sentiment_subjectivity']].median()
print(sentiment_by_exp.sort_values('sentiment_polarity', ascending=False).round(3))

### Interpretation

- **Most job descriptions are slightly positive** (polarity > 0) — companies use encouraging language
- **Subjectivity is moderate** — mix of factual requirements and aspirational language
- If higher-level roles have higher polarity, it suggests employers use more "selling" language for senior positions
- We'll test whether sentiment correlates with application rate in Phase 3 modeling

## 3. Entry-Level Paradox Analysis (Theme 3)

**The question**: Do jobs labeled "Entry level" actually demand senior-level qualifications?

We detect **senior signals** in job descriptions:
- Years of experience mentioned ("3+ years", "5 years")
- Senior keywords ("senior", "lead", "manage", "expert", "advanced")
- Authority language ("strategic", "proven track record", "extensive experience")

If entry-level postings have high counts of these signals, that's the paradox in action.

In [ ]:
# Detect senior signals in all descriptions
df = add_senior_signals(df, text_col='desc_clean')

print("\nSenior signal stats:")
print(df[['senior_signal_count', 'max_years_required']].describe().round(1))

In [ ]:
# THE KEY ANALYSIS: Senior signals by experience level
print("=" * 60)
print("ENTRY-LEVEL PARADOX: Senior signals by experience level")
print("=" * 60)

paradox = df.groupby('experience_level').agg(
    n_postings=('job_id', 'count'),
    median_signals=('senior_signal_count', 'median'),
    mean_signals=('senior_signal_count', 'mean'),
    pct_mentioning_years=('max_years_required', lambda x: (x > 0).mean() * 100),
    median_years_when_mentioned=('max_years_required', lambda x: x[x > 0].median() if (x > 0).any() else 0),
).round(2)

# Sort by expected seniority
exp_order = ['Internship', 'Entry level', 'Associate', 'Mid-Senior level', 'Director', 'Executive', 'Unknown']
available = []
for e in exp_order:
    if e in paradox.index:
        available.append(e)
paradox = paradox.loc[available]

print(paradox)
print("\n→ If entry-level jobs have high 'pct_mentioning_years' or high 'median_years_when_mentioned',")
print("  that's evidence of the entry-level paradox.")

In [ ]:
# Visualize: Senior signal count by experience level
exp_order_available = []
for e in exp_order:
    if e in df['experience_level'].unique():
        exp_order_available.append(e)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Senior signal counts
sns.boxplot(
    data=df[df['senior_signal_count'] <= 20],
    x='experience_level', y='senior_signal_count',
    order=exp_order_available, ax=axes[0]
)
axes[0].set_title('Senior Signal Count by Experience Level')
axes[0].set_xlabel('Experience Level')
axes[0].set_ylabel('Number of Senior Signals')
axes[0].tick_params(axis='x', rotation=45)

# Max years required
df_with_years = df[df['max_years_required'] > 0]
sns.boxplot(
    data=df_with_years[df_with_years['max_years_required'] <= 15],
    x='experience_level', y='max_years_required',
    order=exp_order_available, ax=axes[1]
)
axes[1].set_title('Years of Experience Required (When Mentioned)')
axes[1].set_xlabel('Experience Level')
axes[1].set_ylabel('Max Years Mentioned')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
save_fig(fig, 'nlp_entry_level_paradox')
plt.show()

### Entry-Level Paradox Findings

Look at the data above and answer:
1. What percentage of entry-level jobs mention specific years of experience?
2. When entry-level jobs DO mention years, what's the median? (If it's 2-3+, that's paradoxical)
3. Do entry-level jobs have more senior signals than you'd expect?

This is one of the most discussed issues in the job market — and now you have **data** to back it up.
We'll formalize this into a classifier in Phase 3.

## 4. Join Structured Skill Data

LinkedIn provides structured skill tags for some jobs (`job_skills.csv`).
Let's join these to enrich our feature set.

In [ ]:
# Load structured skill data
if JOB_SKILLS_CSV.exists() and SKILLS_MAP_CSV.exists():
    job_skills = pd.read_csv(JOB_SKILLS_CSV)
    skills_map = pd.read_csv(SKILLS_MAP_CSV)
    
    print(f"Job-skill links: {len(job_skills):,} rows")
    print(f"Skill categories: {len(skills_map)} unique skills")
    print(f"\nSkill mapping:")
    print(skills_map.to_string(index=False))
else:
    print("Skill data files not found — skipping structured skills.")
    job_skills = None

In [ ]:
if job_skills is not None:
    # Map abbreviations to full names
    job_skills_named = job_skills.merge(skills_map, on='skill_abr', how='left')
    
    # Count skills per job
    skills_per_job = job_skills_named.groupby('job_id')['skill_name'].apply(list).reset_index()
    skills_per_job.columns = ['job_id', 'skill_list']
    skills_per_job['n_skills'] = skills_per_job['skill_list'].apply(len)
    
    # Join to main dataframe
    df = df.merge(skills_per_job[['job_id', 'n_skills']], on='job_id', how='left')
    df['n_skills'] = df['n_skills'].fillna(0).astype(int)
    
    print(f"Postings with structured skills: {(df['n_skills'] > 0).sum():,} ({(df['n_skills'] > 0).mean()*100:.1f}%)")
    print(f"\nSkills per job (when available):")
    print(df[df['n_skills'] > 0]['n_skills'].describe().round(1))
    
    # Most common skills across all postings
    print("\nTop 15 most tagged skills:")
    top_skills = job_skills_named['skill_name'].value_counts().head(15)
    for skill, count in top_skills.items():
        print(f"  {skill:<30s} {count:>8,}")

### Why structured skills matter

- They give us **ground truth labels** for what skills a job requires
- In Phase 3, we can compare: "Do jobs tagged with 'Engineering' have different salary distributions than 'Sales'?"
- In Phase 4 (RAG), users can ask: "Show me jobs that require Management + Engineering skills"
- The skill count itself (`n_skills`) is a feature — jobs requiring more skills might pay more

## 5. Feature Summary & Save

In [ ]:
# Summary of all features we've created
print("Features created in Phase 2:")
print("=" * 50)

feature_groups = {
    "Text stats (Notebook 04)": ['desc_word_count', 'desc_char_count', 'title_word_count'],
    "Sentiment (Notebook 05)": ['sentiment_polarity', 'sentiment_subjectivity'],
    "Entry-level paradox (Notebook 05)": ['senior_signal_count', 'max_years_required'],
    "Structured skills (Notebook 05)": ['n_skills'],
}

for group, cols in feature_groups.items():
    print(f"\n{group}:")
    available_cols = []
    for c in cols:
        if c in df.columns:
            available_cols.append(c)
    if available_cols:
        print(df[available_cols].describe().round(3).to_string())

In [ ]:
# Save the enriched dataset
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df.to_parquet(POSTINGS_FEATURES_PARQUET, index=False)

print(f"Saved to {POSTINGS_FEATURES_PARQUET}")
print(f"Shape: {df.shape}")
print(f"File size: {POSTINGS_FEATURES_PARQUET.stat().st_size / 1e6:.1f} MB")

print(f"\nNew columns added in this notebook:")
new_cols = ['sentiment_polarity', 'sentiment_subjectivity', 'senior_signal_count', 'max_years_required', 'n_skills']
for col in new_cols:
    if col in df.columns:
        print(f"  - {col}")

## Summary

### What we did
1. **TF-IDF analysis** — found the most distinctive terms per experience level
2. **Sentiment analysis** — measured how positive/subjective job descriptions are
3. **Entry-level paradox** — detected senior signals in entry-level job descriptions
4. **Structured skills** — joined LinkedIn's skill tags to our dataset

### Mental model: Feature engineering

```
Text data → Multiple feature extraction methods → Numeric columns → Ready for ML

desc_clean ──→ TF-IDF ────→ Sparse matrix (for analysis, not saved)
           ├─→ Sentiment ──→ polarity, subjectivity columns
           └─→ Senior signals → signal_count, max_years columns

job_skills.csv → Join on job_id → n_skills column
```

### Key insight: TF-IDF is for analysis, not ML features

We used TF-IDF here to **understand** the data (what terms matter per category).
For actual ML models, we'll use **embeddings** (notebook 06) — they capture meaning,
not just word counts. TF-IDF treats "dog" and "canine" as completely different;
embeddings know they're similar.

---

**→ Next**: Notebook 06 — Embeddings & Topic Modeling (sentence-transformers, BERTopic)